# 实验 5 — 故意破坏一个 dbt test

**目标**：理解 dbt test 失败时的输出，学会从报错中诊断数据质量问题。

这个实验会**故意让测试失败**，然后恢复。不会改坏任何核心数据。

In [ ]:
import subprocess, os

def run_dbt(cmd: list[str]) -> str:
    result = subprocess.run(
        ["uv", "run", "dbt"] + cmd,
        capture_output=True, text=True, cwd="../dbt_project",
        env={**os.environ, "DBT_PROFILES_DIR": "."}
    )
    return result.stdout + result.stderr

print("当前 test 全部通过：")
out = run_dbt(["test", "--select", "dim_currencies"])
# 只打最后几行
print("\n".join(out.strip().splitlines()[-6:]))

## 步骤 1：修改 `_marts.yml`，把 USD 从 accepted_values 移除

打开 `dbt_project/models/marts/_marts.yml`，把：
```yaml
values: ['USD', 'GBP', 'CHF', 'JPY', 'CNY']
```
改成：
```yaml
values: ['GBP', 'CHF', 'JPY', 'CNY']
```

然后运行下面的 cell。

In [ ]:
out = run_dbt(["test", "--select", "dim_currencies", "--no-partial-parse"])
print(out)

## 步骤 2：分析失败输出

dbt test 失败时会给出：
- 失败的行数（`Got X results, configured to fail if != 0`）
- 诊断 SQL 路径（`compiled code at ...`），可以直接复制到 DuckDB 查询

In [ ]:
# dbt 把失败测试的查询存在 target/compiled 里，直接用 duckdb 跑它
import duckdb
from pathlib import Path

DB = "../data/sandbox.duckdb"
con = duckdb.connect(DB)

# 手动重现 accepted_values 失败的行
sql = """
SELECT currency_code, COUNT(*) as cnt
FROM main.dim_currencies
WHERE currency_code NOT IN ('GBP', 'CHF', 'JPY', 'CNY')  -- 假设 USD 被排除了
GROUP BY 1
"""
con.execute(sql).df()

## 步骤 3：恢复 `_marts.yml`，把 USD 加回来，重跑确认全绿

In [ ]:
out = run_dbt(["test", "--select", "dim_currencies", "--no-partial-parse"])
print("\n".join(out.strip().splitlines()[-6:]))